## configs

In [10]:
import importlib
import utils.config_sanitize
import utils.run_io
importlib.reload(utils.config_sanitize)
importlib.reload(utils.run_io)

<module 'utils.run_io' from '/home/alessio/Scrivania/Alessio/Tesi/Tacconi/EnergyTSF/utils/run_io.py'>

In [11]:
from argparse import Namespace
import torch
import torch.nn as nn
import torch.optim as optim
from engine.seed import set_global_seed
from data_provider.data_factory import data_factory
from models.mamba import MambaModel
from engine.trainer import Trainer
from utils.exp_utils import build_run_paths_from_configs




# --- CONFIGURATION CONSOLE ---
configs = Namespace(
    # -- Experiment --
    model_name="mamba",
    token_style="variate",   # "time" | "variate"
    seed=42,
    verbose=True,

    # -- Data --
    data_name="ETTh1",       # "ETTh1" | "ECL"
    window=96,               # L
    horizon=96,              # T
    stride=1,
    batch_size=32,
    ratios=(0.7, 0.15, 0.15),
    

    # -- Model Hparams --
    channels=7,              # V (ETTh1=7, ECL=321)
    d_model=64,             # H or D 
    num_layers=2,
    dropout=0.1,

    # -- Mamba Params --
    d_state=2,   # 2
    expand=2,
    d_conv=4,
    dt_rank="auto",
    dt_min=0.001,
    dt_max=0.1,
    use_fast_path=True,
    bidirectional=True,      # (Only toggleable in Time mode)

    # -- FFN Params --
    d_ff=256,
    activation="gelu",       # "relu" | "gelu" | "silu"
    
    # -- Time Token Extras (Ignored if variate) --
    pooling="attention",     # "last" | "attention"
    temperature=1.0,

    # -- Optimization --
    lr=7e-5,
    weight_decay=1e-5,
    scheduler_factor=0.5,
    scheduler_patience=3,
    min_lr=1e-7,
    early_stop_patience=10,
    num_epochs=20,
    device="cuda" if torch.cuda.is_available() else "cpu",

)


## Load Data and Model

In [12]:
set_global_seed(seed=configs.seed, verbose=False)

dm = data_factory(
    data_name=configs.data_name,
    window=configs.window,
    horizon=configs.horizon,
    stride=configs.stride,
    batch_size=configs.batch_size,
    ratios=configs.ratios,
    verbose=True,
)
train_loader, val_loader, test_loader = dm.setup(seed=configs.seed, verbose=True,)

model = MambaModel(configs=configs, verbose=True,).to(configs.device)


--- 📊 DataModule: ETTh1 ---
Window: 96 | Horizon: 96 | Stride: 1
Batch Size: 32 | Ratios: (0.7, 0.15, 0.15)
🧩 Created raw splits:
   Train: X=(12194, 7), y=(12194, 7)
   Val:   X=(2613, 7), y=(2613, 7)
   Test:  X=(2613, 7), y=(2613, 7)
🔁 Created windowed data: X_seq (12003, 96, 7), y_seq (12003, 96, 7)
🔁 Created windowed data: X_seq (2422, 96, 7), y_seq (2422, 96, 7)
🔁 Created windowed data: X_seq (2422, 96, 7), y_seq (2422, 96, 7)
📦 Train batch → X: torch.Size([32, 96, 7]), Y: torch.Size([32, 96, 7])

--- 🤖 Model: MambaModel Ready ---
Family: mamba | Token Style: variate
Common: channels=7 | d_model=64 | layers=2 | dropout=0.1
Mamba: d_state=2 | expand=2 | d_conv=4 | bidirectional=True
Mamba FFN: d_ff=256 | act=gelu
Params: 100,320 | Size: 0.38 MB



## Training Logic

In [13]:
paths = build_run_paths_from_configs(
    configs=configs,
    seed=configs.seed, 
    root_dir="runs",
    verbose=True,
)

save_run_dir = paths["run_root"]
save_artifacts_dir = paths["train_dir"]
save_test_dir = paths["test_dir"]




[exp_utils] Run directories created:
  run_root : runs/ETTh1/variate/mamba/L96_T96/seed_42/2026-03-06_23-36
  train_dir: runs/ETTh1/variate/mamba/L96_T96/seed_42/2026-03-06_23-36/train
  test_dir : runs/ETTh1/variate/mamba/L96_T96/seed_42/2026-03-06_23-36/test



In [14]:
optimizer = optim.AdamW(
    params=model.parameters(),
    lr=configs.lr,
    weight_decay=configs.weight_decay,
)

criterion = nn.MSELoss()

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer=optimizer,
    mode="min",
    factor=configs.scheduler_factor,
    patience=configs.scheduler_patience,
)


trainer = Trainer(
    model=model,
    optimizer=optimizer, 
    criterion=criterion, 
    scheduler=scheduler,
    device=configs.device, 
    patience=5, 
    save_artifacts_dir=save_artifacts_dir,
    save_run_dir=save_run_dir, 
    run_config=configs, 
    verbose=True,
)

history = trainer.fit(
    train_loader=train_loader, 
    val_loader=val_loader, 
    num_epochs=configs.num_epochs,
    trial=None,
    save_all=True,
    verbose=True,
)

🧾 Run summary saved to runs/ETTh1/variate/mamba/L96_T96/seed_42/2026-03-06_23-36/run_summary.json
Starting training... Checkpoints: runs/ETTh1/variate/mamba/L96_T96/seed_42/2026-03-06_23-36/train
Epoch 00 | Tr_MSE: 0.7533 | Val_MSE: 0.5526
Epoch 01 | Tr_MSE: 0.5290 | Val_MSE: 0.4825
Epoch 02 | Tr_MSE: 0.4584 | Val_MSE: 0.4389
Epoch 03 | Tr_MSE: 0.4141 | Val_MSE: 0.4058
Epoch 04 | Tr_MSE: 0.3888 | Val_MSE: 0.3902
Epoch 05 | Tr_MSE: 0.3743 | Val_MSE: 0.3844
Epoch 06 | Tr_MSE: 0.3654 | Val_MSE: 0.3872
Epoch 07 | Tr_MSE: 0.3585 | Val_MSE: 0.3824
Epoch 08 | Tr_MSE: 0.3524 | Val_MSE: 0.3828
Epoch 09 | Tr_MSE: 0.3475 | Val_MSE: 0.3762
Epoch 10 | Tr_MSE: 0.3430 | Val_MSE: 0.3765
Epoch 11 | Tr_MSE: 0.3390 | Val_MSE: 0.3792
Epoch 12 | Tr_MSE: 0.3352 | Val_MSE: 0.3758
Epoch 13 | Tr_MSE: 0.3317 | Val_MSE: 0.3762
Epoch 14 | Tr_MSE: 0.3285 | Val_MSE: 0.3785
Epoch 15 | Tr_MSE: 0.3252 | Val_MSE: 0.3758
Epoch 16 | Tr_MSE: 0.3228 | Val_MSE: 0.3734
Epoch 17 | Tr_MSE: 0.3197 | Val_MSE: 0.3764
Epoch 18 | T